# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations and role-based column masking on sample records using PySpark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, trunc, udf
from pyspark.sql.types import DateType
import datetime

spark = SparkSession.builder.appName("Day09_Security_Marts").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Mock Silver Data
data = [
    (1, "John Doe", "123-45-6789", datetime.date(1980, 5, 15)),
    (2, "Jane Smith", "987-65-4321", datetime.date(1992, 11, 2))
]
schema = ["person_id", "name", "ssn", "dob"]
df_silver = spark.createDataFrame(data, schema)
print("Original Silver Data:")
df_silver.show()

In [ ]:
# Simulate Role-Based Masking
def apply_masking(df, role):
    if role == "ehdip_data_scientist":
        # Mask SSN, Truncate DOB to Year
        return df \
            .withColumn("ssn", lit("***-**-****")) \
            .withColumn("dob", trunc(col("dob"), "year"))
    elif role == "ehdip_clinical_analyst":
        # Mask SSN, keep DOB
        return df.withColumn("ssn", lit("***-**-****"))
    else: # ehdip_data_engineer
        return df

print("View as Data Scientist:")
apply_masking(df_silver, "ehdip_data_scientist").show()

print("View as Clinical Analyst:")
apply_masking(df_silver, "ehdip_clinical_analyst").show()

print("View as Data Engineer:")
apply_masking(df_silver, "ehdip_data_engineer").show()